# New fold seeds for the nested ask-rule evaluation (Kaggle)

The same run as `seeds.ipynb`, for Kaggle. The bundle is attached as a **private** dataset. Results are written to `/tmp`
and zipped into `/kaggle/working/seeds_results.zip` after every stage; that zip is the only output file.

**Settings:** Accelerator None (CPU), Internet On (to install scikit-learn 1.9.1).
**Run it with Save Version › Save & Run All (Commit)**, then close the browser. About 4–6 hours. The run stops by itself
within the first half hour if this environment does not reproduce the original fits.

**To continue an interrupted run:** Add Input › Notebook Output › the earlier version of this notebook (its output holds
`seeds_results.zip`), then Save & Run All again. Finished fits are skipped.

## 1. Find the bundle, and an earlier run if one is attached

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path
INPUT, WORK, TMP = Path('/kaggle/input'), Path('/kaggle/working'), Path('/tmp')
ONLY = None  # every target; tests use a smaller set

def find_bundle():
    hits = sorted(INPUT.rglob('seeds_lib.py'))
    if hits:
        return hits[0].parent
    for z in sorted(INPUT.rglob('*.zip')):  # a zip that Kaggle did not unpack
        with zipfile.ZipFile(z) as f:
            names = [n for n in f.namelist() if n.endswith('seeds_lib.py')]
            if names:
                f.extractall(TMP / 'bundle')
                return (TMP / 'bundle' / names[0]).parent
    raise FileNotFoundError('bundle not found: Add Input > Your Datasets > the dataset that holds seeds_lib.py')

BASE = str(find_bundle())
RESULTS = str(TMP / 'seeds_results')
for f in ('config.json', 'targets.csv', 'frozen_seed0.csv', 'nested_lib.py', 'seeds_lib.py', 'data'):
    assert os.path.exists(os.path.join(BASE, f)), f'missing {f} in {BASE}'
earlier = sorted(INPUT.rglob('seeds_results.zip'))
if earlier and not os.path.exists(RESULTS):
    with zipfile.ZipFile(earlier[0]) as f:
        f.extractall(RESULTS)
    print('continuing from', earlier[0], '-', len(list(Path(RESULTS).rglob('*.json'))), 'files')
print('bundle:', BASE)

## 2. Install scikit-learn 1.9.1 (needs Internet On)

In [ ]:
%pip install -q "scikit-learn==1.9.1"

## 3. Load the library and check the version and the tables

Stops if scikit-learn is not 1.9.1 or a table is damaged.

In [ ]:
import importlib
sys.dont_write_bytecode = True
sys.path.insert(0, BASE)
import nested_lib, seeds_lib
importlib.reload(nested_lib); importlib.reload(seeds_lib)
cfg, targets = nested_lib.load_bundle(BASE)
nested_lib.check_sklearn(cfg)
print(seeds_lib.verify_data(BASE), 'tables match their checksums')
print(nested_lib.versions(), os.cpu_count(), 'CPUs')

def n_check(only):
    t = targets if only is None else targets[targets.instance_id.isin(only)]
    return int(t.arms.str.split('|').str.len().sum()) * len(cfg['check_fits'])

def verify(long, chk, expected, allowed):
    nested = long[(long.design == 'nested') & (long.decision == 'native')]
    assert nested.eval_rows_in_train.fillna(0).sum() == 0, 'evaluation rows in a nested fit: stop and send this output'
    diff = chk[[c for c in chk.columns if c.startswith('diff_')]].fillna(0).max(axis=1)
    same = int((diff < 1e-9).sum())
    assert len(chk) == expected, f'{len(chk)} of {expected} check fits found'
    assert same >= expected - allowed, f'only {same}/{expected} fits reproduce the original run: stop and send this output'
    print(f'check passed: {same}/{expected} identical')

def save():
    out = seeds_lib.summarize(RESULTS, BASE)
    shutil.make_archive(str(WORK / 'seeds_results'), 'zip', RESULTS)
    print('saved', WORK / 'seeds_results.zip', flush=True)
    return out

count = [0]
def quiet(msg):  # every 50th finished fit, and every fit that is not ok
    count[0] += 1
    if ': ok (' not in msg or count[0] % 50 == 0:
        print(msg, flush=True)

## 4. Smoke test (about a minute)

I010 is a target whose fits depend on the scikit-learn version: all 10 check fits must be identical.

In [ ]:
smoke = str(TMP / 'smoke_seeds')
seeds_lib.run_check(BASE, smoke, only={'I010'}, log=quiet)
seeds_lib.run(BASE, smoke, 1, only={'I010'}, log=quiet)
long, status, chk = seeds_lib.summarize(smoke, BASE)
verify(long, chk, n_check({'I010'}), allowed=0)

## 5. Check against the original fits (about 25 min)

Seed 0 is not a result: one nested fold and one evaluation fold of the original assignment, refitted. At least 348 of
the 350 fits must be identical (the October check had one fit out of 175 that differed slightly).

In [ ]:
seeds_lib.run_check(BASE, RESULTS, only=ONLY, log=quiet)
long, status, chk = save()
verify(long, chk, n_check(ONLY), allowed=2)

## 6. Fold seeds 1 to 5 (about 1 hour each; the zip is updated after every seed)

In [ ]:
for s in cfg['fold_seeds']:
    print(f'=== fold seed {s} ===', flush=True)
    seeds_lib.run(BASE, RESULTS, s, only=ONLY, retry_errors=True, log=quiet)
    save()

## 7. Summary

In [ ]:
long, status, chk = save()
if ONLY is None and not status.complete.all():
    print('WARNING: some seeds are not complete (table above). Send this output.')
print('done: download seeds_results.zip from the Output tab of this version')